In [1]:
import pandas as pd 
import numpy as np
import joblib
import time
import sklearn.metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

## Load dataset

In [2]:
def calculate_macro_tpr_fpr(voting_cm):
    num_classes = voting_cm.shape[0]
    tpr_list = []
    fpr_list = []

    for i in range(num_classes):
        TP = voting_cm[i, i]
        FN = np.sum(voting_cm[i, :]) - TP
        FP = np.sum(voting_cm[:, i]) - TP
        TN = np.sum(voting_cm) - (TP + FN + FP)

        TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
        FPR = FP / (FP + TN) if (FP + TN) > 0 else 0

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    macro_tpr = np.mean(tpr_list)
    macro_fpr = np.mean(fpr_list)

    return macro_tpr, macro_fpr

train = pd.read_csv('./dataset/FDM_augmented_9600.csv')
test = pd.read_csv('./test_shap_65.csv')

In [3]:
X_train = train.drop(['Label'], axis=1)
y_train = train['Label']
X_test = test.drop(['Label'], axis=1)
y_test = test['Label']
# label_encoder = LabelEncoder()
# y_train = label_encoder.fit_transform(y_train)
# y_test = label_encoder.transform(y_test)

In [4]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

## XGBoost

In [10]:
num_classes = len(np.unique(y_train))
xgb_params = {
    'tree_method': 'hist',
    'device': 'cpu',
    'max_depth': 10,
    'n_estimators': 2500,
    'learning_rate': 0.55,
    'eval_metric': 'mlogloss',
    'objective': 'multi:softprob',
    'num_class': num_classes,
    'booster': 'gbtree',
    'random_state': 42,
    'verbosity': 0,
    'n_jobs': -1
}

print("XGBClassifier Starting")
xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_start_time = time.time()
xgb_prediction = xgb_model.predict(X_test)
xgb_end_time = time.time()
xgb_time = xgb_end_time - xgb_start_time
print("XGBClassifier Finished")

xgb_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=xgb_prediction)
xgb_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=xgb_prediction, average='macro')
xgb_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=xgb_prediction)

print("XGBoost report:")
print("XGBoost Time:", xgb_time)
print("XGBoost Accuracy:", xgb_acc)
print("XGBoost Precision:", xgb_precision)
print("XGBoost F1:", xgb_f1)
print("XGBoost Recall:", xgb_recall)
print("XGBoost CM:\n", xgb_cm)
xgb_tpr, xgb_fpr = calculate_macro_tpr_fpr(xgb_cm)
print(f'XGBoost Macro-average TPR: {xgb_tpr}')
print(f'XGBoost Macro-average FPR: {xgb_fpr}')

XGBClassifier Starting
XGBClassifier Finished
XGBoost report:
XGBoost Time: 0.059983015060424805
XGBoost Accuracy: 0.8427021696252466
XGBoost Precision: 0.835864652218615
XGBoost F1: 0.836413661275019
XGBoost Recall: 0.8427021696252465
XGBoost CM:
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   1 168   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  35   1   3   0   0  58  47   8  16   0]
 [  0   0   0   0   0   0   0  45 124   0   0   0]
 [  0   0  11   0   0   0   0   6   0  91  61   0]
 [  0   0   0   0   0   0   0   0   0  84  85   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
XGBoost Macro-average TPR: 0.8427021696252465
XGBoost Macro-average FPR: 0.014299802761341221


## ExtraTree

In [11]:
et_params = {
    "n_estimators": 178,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("ExtraTreesClassifier Starting")
et_model = ExtraTreesClassifier(**et_params)
et_model.fit(X=X_train, y=y_train)
et_start_time = time.time()
et_prediction = et_model.predict(X_test)
et_end_time = time.time()
et_time = et_end_time - et_start_time
print("ExtraTreesClassifier Finished")

et_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=et_prediction)
et_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=et_prediction, average='macro')
et_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=et_prediction)
et_fp = et_cm[0, 1]
print("ExtraTrees report:")
print("ExtraTrees Time:", et_end_time - et_start_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

ExtraTreesClassifier Starting
ExtraTreesClassifier Finished
ExtraTrees report:
ExtraTrees Time: 0.20043134689331055
ExtraTrees Accuracy: 0.8574950690335306
ExtraTrees Precision: 0.8554981511691016
ExtraTrees F1: 0.852295322542362
ExtraTrees Recall: 0.8574950690335306
ExtraTrees FP: 0
ExtraTrees CM:
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   0 168   1   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  26   0   0   0   0  73  43  10  16   0]
 [  0   0   0   0   0   0   0  31 138   0   0   0]
 [  0   0  12   0   0   0   0   5   0 106  46   0]
 [  0   0   0   0   0   0   0   0   0  98  71   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
ExtraTrees Macro-average TPR: 0.8574950690335306
ExtraTrees Macro-average FPR: 0.01295

In [5]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
et_params = {
    "n_estimators": 70,        # sẽ thay trong vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 170, 190, 1  # có thể đổi STEP=1 để quét mịn

print("ExtraTreesClassifier Sweep Starting")

records = []
best_key = None
best_model, best_cm, best_pred = None, None, None
best_n, best_time = None, None

for n in range(START, END + 1, STEP):
    params = et_params.copy()
    params["n_estimators"] = n

    model = ExtraTreesClassifier(**params)
    model.fit(X_train, y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_test, pred)
    precision = sklearn.precision_score(y_test, pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_test, pred, average='macro')
    recall = sklearn.recall_score(y_test, pred, average='macro')
    cm = sklearn.confusion_matrix(y_test, pred)
    et_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": et_fp
    })

    key = (acc, f1, recall, -n)  # tie-break
    if best_key is None or key > best_key:
        best_key = key
        best_model, best_cm, best_pred = model, cm, pred
        best_n, best_time = n, t1 - t0

print("ExtraTreesClassifier Sweep Finished")

# === Tổng hợp kết quả ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], 
    ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === Report tốt nhất ===
et_acc = sklearn.accuracy_score(y_test, best_pred)
et_precision = sklearn.precision_score(y_test, best_pred, average='macro', zero_division=0)
et_f1 = sklearn.f1_score(y_test, best_pred, average='macro')
et_recall = sklearn.recall_score(y_test, best_pred, average='macro')
et_cm = best_cm
et_fp = et_cm[0, 1] if et_cm.shape == (2, 2) else None
et_tpr, et_fpr = calculate_macro_tpr_fpr(et_cm)

print("\n===== Best ExtraTrees report =====")
print(f"Best n_estimators: {best_n}")
print("ExtraTrees Time:", best_time)
print("ExtraTrees Accuracy:", et_acc)
print("ExtraTrees Precision:", et_precision)
print("ExtraTrees F1:", et_f1)
print("ExtraTrees Recall:", et_recall)
print("ExtraTrees FP:", et_fp)
print("ExtraTrees CM:\n", et_cm)
print(f'ExtraTrees Macro-average TPR: {et_tpr}')
print(f'ExtraTrees Macro-average FPR: {et_fpr}')

# df_results.to_csv("./et_sweep_n_estimators_30_500.csv", index=False)


ExtraTreesClassifier Sweep Starting
ExtraTreesClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0           178  0.857495  0.852295      0.857495  0.185296
1           182  0.857002  0.851949      0.857002  0.200108
2           180  0.857002  0.851945      0.857002  0.326133
3           179  0.857002  0.851943      0.857002  0.224113
4           188  0.857002  0.851754      0.857002  0.091191
5           184  0.857002  0.851711      0.857002  0.308639
6           183  0.856509  0.851501      0.856509  0.108341
7           177  0.856509  0.851495      0.856509  0.155093
8           187  0.856509  0.851308      0.856509  0.184985
9           185  0.856509  0.851268      0.856509  0.095685

===== Best ExtraTrees report =====
Best n_estimators: 178
ExtraTrees Time: 0.18529629707336426
ExtraTrees Accuracy: 0.8574950690335306
ExtraTrees Precision: 0.8554981511691016
ExtraTrees F1: 0.852295322542362
ExtraTrees Recall: 

## RandomForest

In [12]:
rf_params = {
    "n_estimators": 981,
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

print("RandomForestClassifier Starting")
rf_model = RandomForestClassifier(**rf_params)
rf_model.fit(X=X_train, y=y_train)
rf_start_time = time.time()
rf_prediction = rf_model.predict(X_test)
rf_end_time = time.time()
rf_time = rf_end_time - rf_start_time
print("RandomForestClassifier Finished")

rf_acc = sklearn.metrics.accuracy_score(y_true=y_test, y_pred=rf_prediction)
rf_precision = sklearn.metrics.precision_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_f1 = sklearn.metrics.f1_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_recall = sklearn.metrics.recall_score(y_true=y_test, y_pred=rf_prediction, average='macro')
rf_cm = sklearn.metrics.confusion_matrix(y_true=y_test, y_pred=rf_prediction)
rf_fp = rf_cm[0, 1]
print("RandomForest report:")
print("RandomForest Time:", rf_end_time - rf_start_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest CM: \n", rf_cm)
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Starting
RandomForestClassifier Finished
RandomForest report:
RandomForest Time: 0.38519978523254395
RandomForest Accuracy: 0.8525641025641025
RandomForest Precision: 0.8491689886682106
RandomForest F1: 0.8470076202425713
RandomForest Recall: 0.8525641025641026
RandomForest CM: 
 [[169   0   0   0   0   0   0   0   0   0   0   0]
 [  0 169   0   0   0   0   0   0   0   0   0   0]
 [  0   0 169   0   0   0   0   0   0   0   0   0]
 [  0   0   0 169   0   0   0   0   0   0   0   0]
 [  0   0   0   0 169   0   0   0   0   0   0   0]
 [  0   0   0   0   0 169   0   0   0   0   0   0]
 [  0   0   0   0   0   0 169   0   0   0   0   0]
 [  0   1  34   1   1   0   2  70  37   9  14   0]
 [  0   0   0   0   0   0   0  35 134   0   0   0]
 [  0   0  11   0   0   0   0   5   0 101  52   0]
 [  0   0   0   0   0   0   0   0   0  97  72   0]
 [  0   0   0   0   0   0   0   0   0   0   0 169]]
RandomForest Macro-average TPR: 0.8525641025641026
RandomForest Macro-average FPR: 

In [10]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics as sklearn

# === Hàm phụ (nếu bạn đã có thì bỏ qua) ===
def calculate_macro_tpr_fpr(cm):
    """
    cm: confusion_matrix (C x C)
    Trả về (macro_TPR, macro_FPR)
    """
    C = cm.shape[0]
    tprs, fprs = [], []
    for c in range(C):
        TP = cm[c, c]
        FN = cm[c, :].sum() - TP
        FP = cm[:, c].sum() - TP
        TN = cm.sum() - TP - FN - FP
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
        tprs.append(tpr)
        fprs.append(fpr)
    return float(np.mean(tprs)), float(np.mean(fprs))

# === Tham số cố định ===
rf_params = {
    "n_estimators": 60,        # sẽ bị thay mỗi vòng lặp
    "max_leaf_nodes": 15000,
    "n_jobs": -1,
    "random_state": 0,
    "bootstrap": True,
    "criterion": "entropy"
}

START, END, STEP = 980, 1000, 1   # có thể đổi STEP=1 nếu muốn quét mịn hơn

print("RandomForestClassifier Sweep Starting")

records = []
best_key = None  # (acc, f1, recall, -n_estimators) để có tie-break rõ ràng
best_model = None
best_cm = None
best_pred = None
best_n = None
best_time = None

for n in range(START, END + 1, STEP):
    params = rf_params.copy()
    params["n_estimators"] = n

    model = RandomForestClassifier(**params)
    model.fit(X=X_train, y=y_train)

    t0 = time.time()
    pred = model.predict(X_test)
    t1 = time.time()

    acc = sklearn.accuracy_score(y_true=y_test, y_pred=pred)
    precision = sklearn.precision_score(y_true=y_test, y_pred=pred, average='macro', zero_division=0)
    f1 = sklearn.f1_score(y_true=y_test, y_pred=pred, average='macro')
    recall = sklearn.recall_score(y_true=y_test, y_pred=pred, average='macro')
    cm = sklearn.confusion_matrix(y_true=y_test, y_pred=pred)
    # Lưu ý: rf_fp = cm[0, 1] chỉ có ý nghĩa khi bài toán nhị phân và label 0/1 nằm đúng trục
    rf_fp = cm[0, 1] if cm.shape == (2, 2) else None
    tpr, fpr = calculate_macro_tpr_fpr(cm)

    records.append({
        "n_estimators": n,
        "time_sec": t1 - t0,
        "accuracy": acc,
        "precision_macro": precision,
        "f1_macro": f1,
        "recall_macro": recall,
        "macro_TPR": tpr,
        "macro_FPR": fpr,
        "fp_(cm01_if_binary)": rf_fp
    })

    # Tie-break: ưu tiên acc -> f1 -> recall; nếu vẫn hòa, chọn n_estimators nhỏ hơn
    key = (acc, f1, recall, -n)
    if (best_key is None) or (key > best_key):
        best_key = key
        best_model = model
        best_cm = cm
        best_pred = pred
        best_n = n
        best_time = t1 - t0

print("RandomForestClassifier Sweep Finished")

# === Tổng hợp kết quả thành DataFrame và in Top-10 theo Accuracy ===
df_results = pd.DataFrame(records).sort_values(
    by=["accuracy", "f1_macro", "recall_macro", "n_estimators"], ascending=[False, False, False, True]
).reset_index(drop=True)

print("\n=== Top-10 cấu hình theo Accuracy ===")
print(df_results.head(10)[["n_estimators", "accuracy", "f1_macro", "recall_macro", "time_sec"]])

# === In report mô hình tốt nhất (giữ format bạn đang dùng) ===
rf_acc = sklearn.accuracy_score(y_true=y_test, y_pred=best_pred)
rf_precision = sklearn.precision_score(y_true=y_test, y_pred=best_pred, average='macro', zero_division=0)
rf_f1 = sklearn.f1_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_recall = sklearn.recall_score(y_true=y_test, y_pred=best_pred, average='macro')
rf_cm = best_cm
rf_fp = rf_cm[0, 1] if rf_cm.shape == (2, 2) else None
rf_tpr, rf_fpr = calculate_macro_tpr_fpr(rf_cm)

print("\n===== Best RandomForest report =====")
print(f"Best n_estimators: {best_n}")
print("RandomForest Time:", best_time)
print("RandomForest Accuracy:", rf_acc)
print("RandomForest Precision:", rf_precision)
print("RandomForest F1:", rf_f1)
print("RandomForest Recall:", rf_recall)
print("RandomForest FP:", rf_fp)
print("RandomForest CM:", rf_cm)
print(f'RandomForest Macro-average TPR: {rf_tpr}')
print(f'RandomForest Macro-average FPR: {rf_fpr}')

RandomForestClassifier Sweep Starting
RandomForestClassifier Sweep Finished

=== Top-10 cấu hình theo Accuracy ===
   n_estimators  accuracy  f1_macro  recall_macro  time_sec
0           981  0.852564  0.847008      0.852564  0.416919
1           987  0.852564  0.847008      0.852564  0.811881
2           988  0.852564  0.847008      0.852564  0.485168
3           989  0.852564  0.847008      0.852564  0.338214
4           990  0.852564  0.847008      0.852564  0.358095
5           991  0.852564  0.847008      0.852564  0.527807
6           992  0.852564  0.847008      0.852564  0.382713
7           993  0.852564  0.847008      0.852564  0.665159
8           994  0.852564  0.847008      0.852564  0.747922
9           995  0.852564  0.847008      0.852564  0.686052

===== Best RandomForest report =====
Best n_estimators: 981
RandomForest Time: 0.4169189929962158
RandomForest Accuracy: 0.8525641025641025
RandomForest Precision: 0.8491689886682106
RandomForest F1: 0.8470076202425713
Rando

In [13]:
joblib.dump(xgb_model, './models/xgb_model_fdm.pkl')
joblib.dump(et_model, './models/et_model_fdm.pkl')
joblib.dump(rf_model, './models/rf_model_fdm.pkl')

print("Models saved to ./models/")
print("  - xgb_model_fdm.pkl")
print("  - et_model_fdm.pkl")
print("  - rf_model_fdm.pkl")

Models saved to ./models/
  - xgb_model_fdm.pkl
  - et_model_fdm.pkl
  - rf_model_fdm.pkl
